In [1]:


from __future__ import annotations

import sys
from pathlib import Path
import numpy as np
import os
os.environ["TF_ENABLE_ONEDNN_OPTS"] = "0"
import tensorflow as tf
from dataclasses import dataclass
import types
from typing import Literal

# -----------------------------------------------------------------------------
# Paths: allow importing both core package and WIP package when running from repo root
# -----------------------------------------------------------------------------
def change_to_repo_root(marker: str = "src") -> None:
    """Change CWD to the repository root (parent of `src`)."""
    here = Path.cwd()
    for parent in [here] + list(here.parents):
        if (parent / marker).is_dir():
            os.chdir(parent)
            break

change_to_repo_root("WIP")
ROOT = Path.cwd()
WIP_SRC = ROOT / "WIP" / "src"
CORE_SRC = ROOT / "src"
WIP= ROOT / "WIP"
sys.path.insert(0, str(WIP_SRC))
sys.path.insert(0, str(CORE_SRC))
sys.path.insert(0, str(WIP))

from Q_Sea_Battle_New.pyr_internal_model_a import PyrInternalModelA
from Q_Sea_Battle_New.pyr_internal_model_b import PyrInternalModelB

from Q_Sea_Battle_New.pyr_measurement_layer_a import PyrMeasurementLayerA
from Q_Sea_Battle_New.pyr_combine_layer_a import PyrCombineLayerA
from Q_Sea_Battle_New.pyr_measurement_layer_b import PyrMeasurementLayerB
from Q_Sea_Battle_New.pyr_combine_layer_b import PyrCombineLayerB

# -----------------------------------------------------------------------------
# Imports (core)
# -----------------------------------------------------------------------------
from Q_Sea_Battle.game_layout import GameLayout
from Q_Sea_Battle.game_env import GameEnv
from Q_Sea_Battle.trainable_assisted_players import TrainableAssistedPlayers
from Q_Sea_Battle.tournament import Tournament
from Q_Sea_Battle.gameplay_adapters import GameplayModelAAdapter, GameplayModelBAdapter

In [2]:

# -----------------------------------------------------------------------------
# Settings (as provided)
# -----------------------------------------------------------------------------
FIELD_SIZE = 4          # 4x4 -> N=16 (requires N a power of 2)
COMMS_SIZE = 1          # Pyramid requires 1 comm bit

P_HIGH = 1.0            # PR-assisted correlation parameter (stochastic mode, 2nd measurement)

GAMES_IN_EVAL_TOURNAMENT = 1000

SEED = 1234

layout_eval = GameLayout(
    field_size=FIELD_SIZE,
    comms_size=COMMS_SIZE,
    number_of_games_in_tournament=GAMES_IN_EVAL_TOURNAMENT,
    channel_noise=0.0,
    enemy_probability=0.5,
)
depth = 4


## Load layer helper

In [3]:
LayerKind = Literal["meas_a", "comb_a", "meas_b", "comb_b"]

@dataclass(frozen=True)
class DiagnoseSettings:
    field_size: int = 4
    seed: int = 1234
    num_games: int = 150_000
    num_samples: int = 2000
    hidden_units: int = 64
    d: int = 0
    weights_dir: Path = Path("WIP/weights_pyr_layers")
    filename_template: str = "{kind}_d{d}.weights.h5"

def weights_path(settings: DiagnoseSettings, kind: LayerKind) -> Path:
    return Path(settings.weights_dir) / settings.filename_template.format(kind=kind, d=settings.d)

def level_sizes(n2: int, d: int) -> tuple[int, int]:
    Ld = n2 // (2 ** d)
    kd = Ld // 2
    return Ld, kd

def wrap_layer_as_model(layer: tf.keras.layers.Layer, kind: LayerKind, *, n2: int, d: int) -> tf.keras.Model:
    Ld, kd = level_sizes(n2, d)
    if kind in ("meas_a", "meas_b"):
        inp = tf.keras.Input(shape=(Ld,), dtype=tf.float32)
        out = layer(inp)
        return tf.keras.Model(inp, out)
    if kind == "comb_a":
        f = tf.keras.Input(shape=(Ld,), dtype=tf.float32)
        o = tf.keras.Input(shape=(kd,), dtype=tf.float32)
        out = layer(f, o)
        return tf.keras.Model([f, o], out)
    if kind == "comb_b":
        g = tf.keras.Input(shape=(Ld,), dtype=tf.float32)
        o = tf.keras.Input(shape=(kd,), dtype=tf.float32)
        c = tf.keras.Input(shape=(1,), dtype=tf.float32)
        out = layer(g, o, c)
        return tf.keras.Model([g, o, c], out)
    raise ValueError(kind)


def load_layer(kind: LayerKind, settings: DiagnoseSettings, *, n2: int) -> tf.keras.layers.Layer:


    if kind == "meas_a":
        layer = PyrMeasurementLayerA(hidden_units=settings.hidden_units)
    elif kind == "comb_a":
        layer = PyrCombineLayerA(hidden_units=settings.hidden_units)
    elif kind == "meas_b":
        layer = PyrMeasurementLayerB(hidden_units=settings.hidden_units)
    elif kind == "comb_b":
        layer = PyrCombineLayerB(hidden_units=settings.hidden_units)
    else:
        raise ValueError(kind)

    model = wrap_layer_as_model(layer, kind, n2=n2, d=settings.d)
    # build (Keras 3: avoid tf.* ops on KerasTensor inputs)
    Ld, kd = level_sizes(n2, settings.d)
    if kind in ("meas_a", "meas_b"):
        _ = model(tf.zeros((1, Ld), tf.float32), training=False)
    elif kind == "comb_a":
        _ = model([tf.zeros((1, Ld), tf.float32), tf.zeros((1, kd), tf.float32)], training=False)
    elif kind == "comb_b":
        _ = model([tf.zeros((1, Ld), tf.float32), tf.zeros((1, kd), tf.float32), tf.zeros((1, 1), tf.float32)], training=False)
    else:
        raise ValueError(kind)

    wp = weights_path(settings, kind)
    if not wp.exists():
        raise FileNotFoundError(f"Missing weights for {kind} at: {wp}")
    model.load_weights(str(wp))
    return layer

## Load trained layers from file

In [4]:


meas_a_layers = []
for d in range(4):
    settings = DiagnoseSettings(
        d=d,
        weights_dir=Path("WIP/weights_pyr_layers"),
        filename_template="{kind}_d{d}.weights.h5"
    )
    layer = load_layer("meas_a", settings, n2=FIELD_SIZE**2)
    meas_a_layers.append(layer)

meas_b_layers = []
for d in range(4):
    settings = DiagnoseSettings(
        d=d,
        weights_dir=Path("WIP/weights_pyr_layers"),
        filename_template="{kind}_d{d}.weights.h5"
    )
    layer = load_layer("meas_b", settings, n2=FIELD_SIZE**2)
    meas_b_layers.append(layer)

comb_a_layers = []
for d in range(4):
    settings = DiagnoseSettings(
        d=d,
        weights_dir=Path("WIP/weights_pyr_layers"),
        filename_template="{kind}_d{d}.weights.h5"
    )
    layer = load_layer("comb_a", settings, n2=FIELD_SIZE**2)
    comb_a_layers.append(layer)

comb_b_layers = []
for d in range(4):
    settings = DiagnoseSettings(
        d=d,
        weights_dir=Path("WIP/weights_pyr_layers"),
        filename_template="{kind}_d{d}.weights.h5"
    )
    layer = load_layer("comb_b", settings, n2=FIELD_SIZE**2)
    comb_b_layers.append(layer)


In [7]:

print("Composing internal models from trained layers...")
internal_a = PyrInternalModelA(
    layout_eval,
    sr_mode="stochastic",
    p_high=P_HIGH,
    beta=10.0,
    alpha=5.0,
    seed=SEED+10,
    measure_layers=meas_a_layers,
    combine_layers=comb_a_layers,
)
#internal_a.compute_with_internal = types.MethodType(new_compute_with_internal_a, internal_a)
internal_b = PyrInternalModelB(
    layout_eval,
    sr_mode="stochastic",
    p_high=P_HIGH,
    beta=10.0,
    alpha=5.0,
    seed=SEED + 11,
    measure_layers=meas_b_layers,
    combine_layers=comb_b_layers,
)
#internal_b.compute_with_internal = types.MethodType(new_compute_with_internal_b, internal_b)

Composing internal models from trained layers...


## Play evaluation tournament

In [8]:

model_a = GameplayModelAAdapter(internal_model_a=internal_a, beta=10.0, harden_between_levels=True)
model_b = GameplayModelBAdapter(internal_model_b=internal_b, beta=10.0, harden_between_levels=True)



print("Running tournament...")
layout_eval = GameLayout(
    field_size=FIELD_SIZE,
    comms_size=COMMS_SIZE,
    number_of_games_in_tournament=GAMES_IN_EVAL_TOURNAMENT,
    channel_noise=0.0,
    enemy_probability=0.5,
)
env = GameEnv(layout_eval)
players = TrainableAssistedPlayers(layout_eval, model_a=model_a, model_b=model_b)

t = Tournament(game_env=env, players=players, game_layout=layout_eval)
log = t.tournament()

print("Tournament finished.")
mean_reward, std_err = log.outcome()
print(f"Pyramid bootstrap tournament over {layout_eval.number_of_games_in_tournament}: {mean_reward:.4f} ± {std_err:.4f}")
#Running tournament...
#Tournament finished.
#Pyramid bootstrap tournament over 1000: 1.0000 ± 0.000



Running tournament...
Tournament finished.
Pyramid bootstrap tournament over 1000: 1.0000 ± 0.0000
